In [82]:

# !pip install streamlit pyngrok
# !pip install PyPDF2
from openai import OpenAI # Import the OpenAI module
import os
import subprocess
import streamlit as st
import PyPDF2

from openai import OpenAI
import os
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY_GOES_HERE"

app_code = """
import streamlit as st
import PyPDF2
import openai
import os
from openai import OpenAI   # Import the OpenAI module

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def extract_text_from_pdf(file):
    pdf_reader = PyPDF2.PdfReader(file)
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text() + "\\n"
    return text

def ask_agent(memory, text, question):
    memory.append({"role": "user", "content": f"Document Text: {text[:3000]}\\nQuestion: {question}"})
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=memory,
        temperature=0.5,
        max_tokens=300
    )
    answer = response.choices[0].message.content.strip()
    memory.append({"role": "assistant", "content": answer})
    return answer

st.title("Personal Research Assistant with Memory")
st.write("Upload your research paper (PDF) and ask multiple questions. I will remember our conversation!")

if "memory" not in st.session_state:
    st.session_state.memory = []

uploaded_file = st.file_uploader("Choose a PDF file", type="pdf")
if uploaded_file:
    if "text" not in st.session_state:
        st.session_state.text = extract_text_from_pdf(uploaded_file)
        st.success("PDF successfully uploaded and parsed!")

    question = st.text_input("Ask a question about the paper:")
    if question:
        with st.spinner("Reading File..."):
            answer = ask_agent(st.session_state.memory, st.session_state.text, question)
        st.write("**Answer:**", answer)

        st.markdown("---")
        st.write("**Conversation History:**")
        for msg in st.session_state.memory:
            if msg["role"] == "user":
                st.write(f"**You:** {msg['content'].split('Question:')[-1].strip()}")
            else:
                st.write(f"**Assistant:** {msg['content']}")
"""

with open("app.py", "w") as f:
    f.write(app_code)



# import time
# import requests
from pyngrok import ngrok, conf

# Kills existing ngrok processes
subprocess.run("pkill ngrok || true", shell=True)

# This line starts Streamlit
streamlit_cmd = "streamlit run app.py --server.address=0.0.0.0 --server.port=8501"

# Run streamlite in background
process = subprocess.Popen(streamlit_cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Starts ngrok tunnel
ngrok.set_auth_token("NGROK AUTH TOKEN GOES HERE")

# Binds ngrok's reserved domain and port to Streamlit using TLS protocol
try:
    ai_agent_url = ngrok.connect(
        addr=8501,
        bind_tls=True,
        hostname="rosella-notarial-uncategorically.ngrok-free.dev"
    )
except Exception as e:
    print("NGROK ERROR:")
    print(e)
    raise SystemExit

ai_agent_url

<NgrokTunnel: "https://rosella-notarial-uncategorically.ngrok-free.dev" -> "http://localhost:8501">